#### Feature engineering

In [1]:
import pandas as pd
import numpy as np

history = pd.read_csv('../data/processed/history_clean.csv')
labs =  pd.read_csv('../data/processed/laboratory_clean.csv', parse_dates=['timestamp'])
patients =   pd.read_csv('../data/processed/patients_clean.csv', parse_dates=['registration_date'])
outcome =  pd.read_csv('../data/processed/outcomes_clean.csv', parse_dates=['diagnosis_time'])
vitals =  pd.read_csv('../data/processed/vitals_clean.csv', parse_dates=['timestamp'])

tables = {'history': history, 'patients': patients, 'labs': labs, 'outcome':outcome, 'vitals' :vitals}

for name, df in tables.items():
    print(f'{name:10s} {df.shape}')


history    (1449, 6)
patients   (600, 5)
labs       (2430, 8)
outcome    (600, 6)
vitals     (11807, 8)


In [14]:
vital_cols = ['heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure']
labs_cols = ['white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count']

In [15]:
rng = np.random.default_rng(7)

def get_prediction_time(row, vitals_df):
    if row['sepsis_event']:
        return row['diagnosis_time'] - pd.Timedelta(hours=9)
    pv = vitals_df[vitals_df['patient_id']== row['patient_id']]
    start, end = pv['timestamp'].min(), pv['timestamp'].max()
    span_hours = max((end - start).total_seconds() / 3600, 1)
    offset = rng.uniform(0.4, 0.9) * span_hours
    return start + pd.Timedelta(hours=offset)

outcomes = outcome.copy()
outcomes['prediction_time'] = outcome.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time',
           'prediction_time']].head(10)

,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,False,NaT,2024-10-21 22:31:02.552633471
1,2,False,NaT,2025-06-26 10:29:09.302720280
2,3,False,NaT,2024-03-01 02:25:35.501601845
3,4,False,NaT,2024-08-02 16:10:19.745594826
4,5,False,NaT,2024-07-10 00:00:48.032376176
5,6,False,NaT,2024-10-17 12:01:52.909136091
6,7,False,NaT,2025-12-17 07:05:33.790790426
7,8,False,NaT,2024-08-30 14:15:42.264998141
8,9,True,2024-06-01 22:58:00,2024-06-01 13:58:00.000000000
9,10,False,NaT,2024-03-31 10:06:58.001875856


In [17]:
LOOKBACK_HOURS = 6

def vital_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp']<=cutoff)&
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in vital_cols:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_min'] = vals.min()
        feats[f'{col}_max'] = vals.max()
        feats[f'{col}_std'] = vals.std() if len(vals) >1 else 0.0
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan

        if len(window) > 1:
            hours = (window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]).total_seconds()/ 3600
            feats[f'{col}_rate_per_hr'] = (vals.iloc[0] - vals.iloc[0]) / hours if hours > 0 else 0.0
        else:
            feats[f'{col}_rate_per_hr'] = 0.0
    return feats

                    

In [18]:
vital_features_rows = [
    {'patient_id': pid, **vital_features(pid, cutoff, vitals)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
vital_features_df = pd.DataFrame(vital_features_rows)
vital_features_df.head(10)

,patient_id,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hr,temperature_mean,temperature_min,temperature_max,...,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hr,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hr
0,1,84.200000,84.2,84.2,0.000000,84.2,0.0,36.800000,36.80,36.80,...,14.3,0.000000,14.3,0.0,114.800000,114.8,114.8,0.000000,114.8,0.0
1,2,87.400000,87.4,87.4,0.000000,87.4,0.0,37.540000,37.54,37.54,...,17.6,0.000000,17.6,0.0,114.500000,114.5,114.5,0.000000,114.5,0.0
2,3,75.300000,72.5,78.1,3.959798,78.1,0.0,36.435000,36.37,36.50,...,15.5,0.353553,15.5,0.0,130.250000,125.6,134.9,6.576093,125.6,0.0
3,4,72.500000,72.5,72.5,0.000000,72.5,0.0,36.490000,36.49,36.49,...,16.4,0.000000,16.4,0.0,119.000000,119.0,119.0,0.000000,119.0,0.0
4,5,74.233333,69.9,79.3,3.149391,69.9,0.0,36.826667,36.71,36.94,...,13.9,0.760044,12.9,0.0,138.250000,128.8,148.6,7.109923,128.8,0.0
5,6,78.900000,78.9,78.9,0.000000,78.9,0.0,37.010000,37.01,37.01,...,22.0,0.000000,22.0,0.0,123.200000,123.2,123.2,0.000000,123.2,0.0
6,7,64.300000,64.3,64.3,0.000000,64.3,0.0,37.520000,37.52,37.52,...,15.0,0.000000,15.0,0.0,121.100000,121.1,121.1,0.000000,121.1,0.0
7,8,80.233333,76.9,83.7,3.401960,80.1,0.0,36.610000,36.23,37.04,...,15.3,0.556776,14.2,0.0,124.733333,120.8,128.1,3.682843,128.1,0.0
8,9,73.000000,69.6,76.4,4.808326,76.4,0.0,37.215000,37.09,37.34,...,17.2,2.121320,17.2,0.0,128.150000,120.1,136.2,11.384419,120.1,0.0
9,10,84.950000,82.5,87.4,3.464823,82.5,0.0,36.195000,35.85,36.54,...,17.4,2.192031,17.4,0.0,116.450000,114.2,118.7,3.181981,118.7,0.0


In [26]:
#Lab lookback 
LAB_LOOKBACK_HOURS = 8

def lab_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp']<=cutoff)&
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id']==pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in labs_cols:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan

    
    return feats


In [28]:
lab_features_rows = [
    {'patient_id': pid, **lab_features(pid, cutoff, labs)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
lab_features_df = pd.DataFrame(lab_features_rows)
lab_features_df.head(10)

,patient_id,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last
0,1,6.815,6.815,8.0,8.0,0.850,0.850,0.78,0.78,301.0,301.0
1,2,7.610,7.610,16.9,16.9,0.540,0.540,0.90,0.90,180.0,180.0
2,3,8.210,8.210,7.6,7.6,1.160,1.160,1.45,1.45,290.0,290.0
3,4,8.680,8.680,3.9,3.9,0.745,0.745,1.34,1.34,283.0,283.0
4,5,4.120,4.120,6.7,6.7,0.970,0.970,0.75,0.75,258.0,258.0
5,6,7.630,7.630,0.5,0.5,0.620,0.620,1.34,1.34,332.0,332.0
6,7,9.250,9.250,4.1,4.1,0.790,0.790,0.61,0.61,233.0,233.0
7,8,9.840,9.840,4.9,4.9,0.690,0.690,0.86,0.86,243.0,243.0
8,9,9.860,9.860,0.5,0.5,1.240,1.240,0.77,0.77,239.0,239.0
9,10,6.220,6.220,4.2,4.2,1.100,1.100,1.03,1.03,298.0,298.0


In [29]:
static = patients[['patient_id','age','gender']].copy()

static['comorbidity_count'] = patients['medical_conditions'].apply(
     lambda x: 0 if pd.isna(x) or x =='Not reported'
     else len(x.split(','))
)
static = pd.get_dummies(static, columns=['gender'], drop_first= True)
static.head()

,patient_id,age,comorbidity_count,gender_Male,gender_Other/Not specified
0,1,66,2,True,False
1,2,42,3,False,False
2,3,74,1,True,False
3,4,77,2,False,False
4,5,25,0,False,False


In [30]:
feature = (
    static
    .merge(vital_features_df, on='patient_id')
    .merge(lab_features_df, on='patient_id')
    .merge(outcome[['patient_id','sepsis_event']], on='patient_id')
)

feature_cols = [
    c for c in feature.columns
    if c not in ('patient_id','sepsis_event')
]

numeric_cols = feature[feature_cols].select_dtypes(include='number').columns

feature[numeric_cols]= feature[numeric_cols].fillna(
    feature[numeric_cols].median()
)
feature['sepsis_event'] = feature['sepsis_event'].astype(int)
print(feature.shape)

(600, 46)


In [33]:
#sanity check
check_cols = ['heart_rate_last','oxygen_saturation_last',
              'crp_last', 'creatinine_last', 'lactate_last'
              ]
feature.groupby('sepsis_event')[check_cols].mean()

,heart_rate_last,oxygen_saturation_last,crp_last,creatinine_last,lactate_last
sepsis_event,,,,,
0,78.249527,97.512879,6.284186,0.905672,0.993049
1,80.506944,97.247917,11.022222,0.915764,1.055833


In [35]:
corr = feature[feature_cols + ['sepsis_event']].corr()['sepsis_event'].drop('sepsis_event')
corr.sort_values(key=abs, ascending=False).head(10)

crp_last                 0.195169
crp_mean                 0.193031
oxygen_saturation_std    0.161821
heart_rate_std           0.140092
blood_pressure_std       0.127917
blood_pressure_last     -0.122969
platelet_count_last     -0.115133
platelet_count_mean     -0.113579
blood_pressure_min      -0.111958
respiratory_rate_last    0.108416
Name: sepsis_event, dtype: float64

In [36]:
feature.to_csv('../data/processed/sepsis_features.csv', index=False)